In [2]:
from hta.trace_analysis import TraceAnalysis
import polars as pl
import pandas as pd
import glob

Couldn't parse gemma train.

In [3]:
dfs = []
for path in glob.glob("/proj/threadtune-PG0/amir/kineto_traces/*"):
    analyzer = TraceAnalysis({0: "device.0.json"}, path)
    rank = 0
    trace_data = analyzer.t.get_trace(rank)
    symbol_table = analyzer.t.symbol_table.get_sym_table()
    dfs.append(
        pl.concat(
            pl.from_pandas(trace_data[trace_data["stream"] != -1])
            .select(pl.lit(rank).alias("rank"), pl.all())
            for rank, trace_data in analyzer.t.traces.items()
        ).join(
            pl.from_dict({"name": list(range(len(symbol_table))), "s_name": symbol_table}),
            on="name",
        )
    )
dfs

Parsed /proj/threadtune-PG0/amir/kineto_traces/deit-train/device.0.json time = 0.24 seconds 
Rounding down ns resolution events due to issue with events overlapping. ts dtype = float64, dur dtype = float64.Please see https://github.com/pytorch/pytorch/pull/122425
Parsed /proj/threadtune-PG0/amir/kineto_traces/deit-train/device.0.json backend=json in 0.48 seconds; current PID:170154
Overall parsing of /proj/threadtune-PG0/amir/kineto_traces/deit-train/device.0.json in 0.54 seconds; current PID:170154
leaving parse_multiple_ranks duration=0.55 seconds
leaving parse_traces duration=0.55 seconds
There is only one iteration in the trace. The analysis result may not be accurate.
Parsed /proj/threadtune-PG0/amir/kineto_traces/gpt2-inference/device.0.json time = 0.03 seconds 
Rounding down ns resolution events due to issue with events overlapping. ts dtype = float64, dur dtype = float64.Please see https://github.com/pytorch/pytorch/pull/122425
Parsed /proj/threadtune-PG0/amir/kineto_traces/gpt

[shape: (1_131, 22)
 ┌──────┬───────┬─────┬──────┬───┬─────────────┬───────────────────┬───────────┬────────────────────┐
 │ rank ┆ index ┆ cat ┆ name ┆ … ┆ external_id ┆ index_correlation ┆ iteration ┆ s_name             │
 │ ---  ┆ ---   ┆ --- ┆ ---  ┆   ┆ ---         ┆ ---               ┆ ---       ┆ ---                │
 │ i32  ┆ i16   ┆ i64 ┆ i64  ┆   ┆ i16         ┆ i16               ┆ i8        ┆ str                │
 ╞══════╪═══════╪═════╪══════╪═══╪═════════════╪═══════════════════╪═══════════╪════════════════════╡
 │ 0    ┆ 13921 ┆ 27  ┆ 117  ┆ … ┆ 7           ┆ 13923             ┆ 0         ┆ Memset (Device)    │
 │ 0    ┆ 13938 ┆ 14  ┆ 53   ┆ … ┆ 7           ┆ 13940             ┆ 0         ┆ void implicit_conv │
 │      ┆       ┆     ┆      ┆   ┆             ┆                   ┆           ┆ olve_sgemm<f…      │
 │ 0    ┆ 13945 ┆ 14  ┆ 13   ┆ … ┆ 10          ┆ 13947             ┆ 0         ┆ void at::native::e │
 │      ┆       ┆     ┆      ┆   ┆             ┆              

In [4]:
df = dfs[0].drop("bytes")

In [5]:
df.filter(pl.col("cat") == 132).select("dur").sum()

dur
f64
0.0


In [6]:
[i for i in symbol_table if i == "trace"]

[]

In [7]:
symbol_table.get_sym_id_map()

AttributeError: 'list' object has no attribute 'get_sym_id_map'

In [ ]:
analyzer.t.symbol_table.get_sym_id_map().get("Trace")

In [8]:
df = pl.read_json("/proj/threadtune-PG0/amir/kineto_traces/bert-train/device.0.json").select(pl.col("traceEvents").explode().struct.unnest())

In [9]:
df.filter(pl.col("cat") == "Trace").select("dur")

dur
f64
652527.152


In [ ]:
df.filter(pl.col("cat") == "Trace").select("dur") - df.filter(pl.col("cat") == "kernel").select("dur").sum()

dur
f64
20745.858
